In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt

/Users/juliasbardelatti/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [5]:
directory = '.'
files = [f for f in os.listdir(directory) if f.endswith('.xlsx') and f != 'combined_cids_demencia.xlsx']

dfs = []
for file in files:
    df = pd.read_excel(os.path.join(directory, file),  decimal=',', thousands='.')
    year = file.split('.')[0] 
    df['ano'] = year
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)
combined_df.to_excel('combined_cids_demencia.xlsx', index=False)

In [ ]:
cid_columns = [col for col in df.columns if 'CID' in col or 'CIAP' in col]
df['total_demencia'] = df[cid_columns].sum(axis=1)

yearly_data = df.groupby('ano').agg({
    'total_demencia': 'sum',
    'populacao_ibge': 'sum'
}).reset_index()

yearly_data['taxa_demencia'] = (yearly_data['total_demencia'] / yearly_data['populacao_ibge']) * 100000

plt.figure(figsize=(10, 6))

data_2019_2024 = yearly_data[yearly_data['ano'] <= 2024]
data_2025 = yearly_data[yearly_data['ano'] == 2025]

plt.plot(data_2019_2024['ano'], data_2019_2024['taxa_demencia'], linestyle='-', color='#114354')

if not data_2025.empty:
    data_2024 = data_2019_2024[data_2019_2024['ano'] == 2024]
    if not data_2024.empty:
        plt.plot([2024, 2025], [data_2024['taxa_demencia'].values[0], data_2025['taxa_demencia'].values[0]],
                 linestyle='--', color='#114354')

for _, row in yearly_data.iterrows():
    plt.text(row['ano'], row['taxa_demencia'], f"{round(row['taxa_demencia'])}",
             ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.title('Taxa de atendimentos individuais por demências (CID-10), por 100 mil habitantes')
plt.xlabel('Ano')
plt.ylabel('Taxa de Demência por 100.000 Habitantes')
plt.ylim(bottom=0, top=1300)

plt.subplots_adjust(bottom=0.25)

plt.figtext(0.5, 0.15, 'Nota: Dados de 2025 são parciais (até julho), por isso a linha está em pontilhado.',
            ha='center', fontsize=10)

plt.figtext(0.5, 0.11, 'Fonte: SISAB e IBGE',
            ha='center', fontsize=9, style='italic')

plt.show()

In [ ]:
uf_yearly = df.groupby(['Uf', 'ano']).agg({
    'total_demencia': 'sum',
    'populacao_ibge': 'sum'
}).reset_index()

uf_yearly['taxa_demencia'] = (uf_yearly['total_demencia'] / uf_yearly['populacao_ibge']) * 100000

data_2024 = uf_yearly[uf_yearly['ano'] == 2024].sort_values('taxa_demencia', ascending=False)
data_2025 = uf_yearly[uf_yearly['ano'] == 2025].sort_values('taxa_demencia', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8))

ax1.bar(data_2024['Uf'], data_2024['taxa_demencia'], color='#114354')
ax1.set_title('Taxa de Demência por UF - 2024')
ax1.set_xlabel('UF')
ax1.set_ylabel('Taxa por 100.000 habitantes')
ax1.tick_params(axis='x', rotation=45)

for i, v in enumerate(data_2024['taxa_demencia']):
    ax1.text(i, v + 5, f"{v:.0f}", ha='center', va='bottom', fontsize=8, fontweight='bold')

ax2.bar(data_2025['Uf'], data_2025['taxa_demencia'], color='#114354')
ax2.set_title('Taxa de Demência por UF - 2025')
ax2.set_xlabel('UF')
ax2.set_ylabel('Taxa por 100.000 habitantes')
ax2.tick_params(axis='x', rotation=45)

for i, v in enumerate(data_2025['taxa_demencia']):
    ax2.text(i, v + 5, f"{v:.0f}", ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()